# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayustar/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: I compare my Week-4 rule against two models: Logistic Regression and Random Forest. Logistic Regression is the simple one I can read and explain. Random Forest is the starter’s strongest model. I keep it only if it clearly beats the simple model and the rule. Complexity alone doesn’t earn a place.
Target: is_down (trend_direction == “down”). This is a proxy for decline from the current window, not a future outcome, so I treat results as directional.
Features: observable numeric columns only. Anything containing trend, label, score, priority, health, flag, or action is excluded.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone https://github.com/Ayustar/flyrank-ml-internship-starter.git
    %cd flyrank-ml-internship-starter

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates("content_id").reset_index(drop=True)
df["is_down"] = (df["trend_direction"] == "down").astype(int)

# features: observable numeric columns, minus anything label-like
banned = ("trend", "label", "is_down", "score", "priority", "health", "flag", "action")
features = [c for c in df.select_dtypes("number").columns
            if c not in ("content_id", "client_id")
            and not any(b in c.lower() for b in banned)]
print(len(features), "features:")
print(features)

# Week-4 baseline score (same formula, no label inputs)
elig = (df["days_since_last_update"] >= 90) & (df["impressions_90d"] >= 500)
stale = (df["days_since_last_update"] / 180).clip(upper=1)
vis = np.log1p(df["impressions_90d"]) / np.log1p(df["impressions_90d"].max())
df["baseline_score"] = np.where(elig, 100 * (0.5 * stale + 0.5 * vis), 0)


Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 148 (delta 56), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.86 MiB | 4.06 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/flyrank-ml-internship-starter
29 features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split: client-holdout. 20% of clients are kept out entirely, so the model is tested on clients it never saw. Pages from the same client share patterns, and a random split would let the model memorise them and make the test too easy. The baseline is scored on the same test pages with the same metrics.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(df, df["is_down"], groups=df["client_id"]))
Xtr, Xte = df.loc[tr, features], df.loc[te, features]
ytr, yte = df.loc[tr, "is_down"].values, df.loc[te, "is_down"].values

print("train:", len(tr), "pages,", df.loc[tr, "client_id"].nunique(), "clients | decline rate", ytr.mean().round(3))
print("test: ", len(te), "pages,", df.loc[te, "client_id"].nunique(), "clients | decline rate", yte.mean().round(3))
print("client overlap:", len(set(df.loc[tr, "client_id"]) & set(df.loc[te, "client_id"])))


train: 23837 pages, 25 clients | decline rate 0.55
test:  6163 pages, 7 clients | decline rate 0.511
client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def p_at_k(y, s, k):
    idx = np.argsort(-s, kind="stable")[:k]
    return y[idx].mean()

lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                   LogisticRegression(max_iter=1000))
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                            n_jobs=-1, random_state=42)
lr.fit(Xtr, ytr)
rf.fit(Xtr.fillna(Xtr.median()), ytr)

scores = {
    "Week-4 baseline rule": df.loc[te, "baseline_score"].values,
    "Logistic Regression": lr.predict_proba(Xte)[:, 1],
    "Random Forest": rf.predict_proba(Xte.fillna(Xtr.median()))[:, 1],
}

rows = []
for name, s in scores.items():
    rows.append({"method": name,
                 "ROC AUC": roc_auc_score(yte, s),
                 "Avg precision": average_precision_score(yte, s),
                 "P@50": p_at_k(yte, s, 50),
                 "P@100": p_at_k(yte, s, 100)})
results = pd.DataFrame(rows).set_index("method").round(3)
print("test decline rate (random-pick precision):", yte.mean().round(3))
print(results)
os.makedirs("work/outputs", exist_ok=True)
results.to_csv("work/outputs/w05_model_vs_baseline.csv")



test decline rate (random-pick precision): 0.511
                      ROC AUC  Avg precision  P@50  P@100
method                                                   
Week-4 baseline rule    0.488          0.502   0.3    0.3
Logistic Regression     0.849          0.873   1.0    1.0
Random Forest           0.942          0.949   1.0    1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# which features matter (on the test set)
pi = permutation_importance(rf, Xte.fillna(Xtr.median()), yte,
                            n_repeats=5, random_state=42, n_jobs=-1,
                            scoring="roc_auc")
imp = pd.Series(pi.importances_mean, index=features).sort_values(ascending=False)
print(imp.head(10).round(4))

# what do the model's top-50 mistakes look like?
te_df = df.loc[te].copy()
te_df["rf_score"] = scores["Random Forest"]
top = te_df.sort_values("rf_score", ascending=False).head(50)
wrong = top[top["is_down"] == 0]
print("\ntop-50 wrong picks:", len(wrong), "of 50")
cols = ["impressions_90d", "sessions_90d", "avg_position", "days_since_last_update", "content_age_days"]
print(pd.DataFrame({"wrong picks (median)": wrong[cols].median(),
                    "right picks (median)": top[top["is_down"] == 1][cols].median(),
                    "all test pages (median)": te_df[cols].median()}).round(1))

impressions_prev_30d     0.3296
impressions_last_30d     0.3027
impressions_90d          0.0262
clicks_last_30d          0.0099
days_with_impressions    0.0088
avg_position             0.0039
sessions_last_30d        0.0038
ctr                      0.0025
scroll_rate              0.0017
sessions_prev_30d        0.0014
dtype: float64

top-50 wrong picks: 0 of 50
                        wrong picks (median)  right picks (median)  \
impressions_90d                          NaN                 329.5   
sessions_90d                             NaN                   3.0   
avg_position                             NaN                   7.0   
days_since_last_update                   NaN                  20.0   
content_age_days                         NaN                 139.5   

                        all test pages (median)  
impressions_90d                           429.0  
sessions_90d                                4.0  
avg_position                                9.8  
days_since_last

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.